In [1]:
import pandas as pd
import numpy as np

# Load all three result files
baseline = pd.read_csv('baseline_results.csv')
rf_xgb = pd.read_csv('rf_xgb_results.csv')
nn_gb = pd.read_csv('nn_gb_results.csv')

print("✓ Loaded Baseline:", len(baseline), "models")
print("✓ Loaded RF+XGBoost:", len(rf_xgb), "models")
print("✓ Loaded NN+GB:", len(nn_gb), "models")
print(f"\nTotal: {len(baseline) + len(rf_xgb) + len(nn_gb)} models")

# Quick preview
print("\nBaseline preview:")
display(baseline[['model', 'accuracy', 'recall_P', 'precision_P', 'f1_P']].head())

print("\nNN+GB preview:")
display(nn_gb[['model', 'accuracy', 'recall_P', 'precision_P', 'f1_P']].head())

✓ Loaded Baseline: 4 models
✓ Loaded RF+XGBoost: 8 models
✓ Loaded NN+GB: 5 models

Total: 17 models

Baseline preview:


,model,accuracy,recall_P,precision_P,f1_P
0,LogisticRegression,0.647103,0.73,0.33,0.46
1,DecisionTree,0.914759,0.86,0.85,0.85
2,GaussianNB,0.482186,0.68,0.24,0.35
3,KNN,0.819150,0.55,0.81,0.66



NN+GB preview:


,model,accuracy,recall_P,precision_P,f1_P
0,GB_orig,0.749310,0.87,0.47,0.61
1,GB_smote,0.783599,0.66,0.71,0.68
2,MLP_orig,0.776417,NaN,NaN,NaN
3,MLP_pca,0.773516,NaN,NaN,NaN
4,MLP_smote,0.758996,NaN,NaN,NaN


In [3]:
# Add category labels
baseline['category'] = 'Baseline'
rf_xgb['category'] = 'RF+XGBoost'
nn_gb['category'] = 'Advanced'

# Add dataset column to baseline (all use original)
if 'dataset' not in baseline.columns:
    baseline['dataset'] = 'Original'

# Add dataset column to nn_gb if missing
if 'dataset' not in nn_gb.columns:
    # Infer from model names
    nn_gb['dataset'] = nn_gb['model'].apply(lambda x: 
        'SMOTE' if 'smote' in x.lower() else 
        'PCA' if 'pca' in x.lower() else 
        'Original'
    )

# Clean up model names if needed
baseline['model'] = baseline['model'].str.replace('LogisticRegression', 'Logistic Regression')
baseline['model'] = baseline['model'].str.replace('DecisionTree', 'Decision Tree')
baseline['model'] = baseline['model'].str.replace('GaussianNB', 'Naive Bayes')

nn_gb['model'] = nn_gb['model'].str.replace('MLP_orig', 'Neural Network')
nn_gb['model'] = nn_gb['model'].str.replace('MLP_pca', 'Neural Network')
nn_gb['model'] = nn_gb['model'].str.replace('MLP_smote', 'Neural Network')
nn_gb['model'] = nn_gb['model'].str.replace('GB_orig', 'Gradient Boosting')
nn_gb['model'] = nn_gb['model'].str.replace('GB_smote', 'Gradient Boosting')

print("✓ Categories and datasets added")

✓ Categories and datasets added


In [23]:
# Common columns for all models
common_cols = ['model', 'dataset', 'category', 'accuracy', 
               'recall_P', 'precision_P', 'f1_P', 
               'recall_macro', 'f1_macro']

# Extract from each dataset
baseline_unified = baseline[['model', 'dataset', 'category', 'accuracy', 
                              'recall_P', 'precision_P', 'f1_P', 
                              'recall_macro', 'f1_macro']].copy()

rf_xgb_unified = rf_xgb[['model', 'dataset', 'category', 'accuracy', 
                          'recall_P', 'precision_P', 'f1_P']].copy()
# Add macro columns if available, otherwise NaN
if 'recall_macro' in rf_xgb.columns:
    rf_xgb_unified['recall_macro'] = rf_xgb['recall_macro']
    rf_xgb_unified['f1_macro'] = rf_xgb['f1_macro']
else:
    rf_xgb_unified['recall_macro'] = np.nan
    rf_xgb_unified['f1_macro'] = np.nan

# NN+GB: Handle missing Poor class metrics for MLP models
nn_gb_unified = nn_gb[['model', 'dataset', 'category', 'accuracy']].copy()

# Add Poor class metrics (NaN for MLP models that don't have them)
nn_gb_unified['recall_P'] = nn_gb['recall_P'] if 'recall_P' in nn_gb.columns else np.nan
nn_gb_unified['precision_P'] = nn_gb['precision_P'] if 'precision_P' in nn_gb.columns else np.nan
nn_gb_unified['f1_P'] = nn_gb['f1_P'] if 'f1_P' in nn_gb.columns else np.nan

# Add note in display
print("Note: MLP models lack per-class Poor metrics - marked as N/A")

# Add macro metrics
nn_gb_unified['recall_macro'] = nn_gb['recall_macro'] if 'recall_macro' in nn_gb.columns else np.nan
nn_gb_unified['f1_macro'] = nn_gb['f1_macro'] if 'f1_macro' in nn_gb.columns else np.nan

print("✓ Common columns extracted")

Note: MLP models lack per-class Poor metrics - marked as N/A
✓ Common columns extracted


In [25]:
# Combine all three dataframes
all_models = pd.concat([baseline_unified, rf_xgb_unified, nn_gb_unified], ignore_index=True)

print("="*80)
print(f"COMBINED: {len(all_models)} MODELS")
print("="*80)
print("\nBreakdown by category:")
print(all_models['category'].value_counts())

print("\nSample of combined data:")
display(all_models.head(10))

COMBINED: 17 MODELS

Breakdown by category:
category
RF+XGBoost    8
Advanced      5
Baseline      4
Name: count, dtype: int64

Sample of combined data:


,model,dataset,category,accuracy,recall_P,precision_P,f1_P,recall_macro,f1_macro
0,Logistic Regression,Original,Baseline,0.647103,0.730000,0.330000,0.460000,0.675024,0.598365
1,Decision Tree,Original,Baseline,0.914759,0.860000,0.850000,0.850000,0.899427,0.897634
2,Naive Bayes,Original,Baseline,0.482186,0.680000,0.240000,0.350000,0.555084,0.395191
3,KNN,Original,Baseline,0.819150,0.550000,0.810000,0.660000,0.743509,0.771776
4,Random Forest,Original (102 features),RF+XGBoost,0.756099,0.869905,0.481388,0.619794,0.790610,0.720605
5,Random Forest,Engineered (130 features),RF+XGBoost,0.754034,0.869569,0.483900,0.621787,0.788804,0.719637
6,Random Forest,SMOTE Balanced,RF+XGBoost,0.744366,0.812451,0.486490,0.608571,0.765766,0.708485
7,Random Forest,PCA (74 components),RF+XGBoost,0.736335,0.806069,0.463897,0.588887,0.758333,0.697595
8,XGBoost,SMOTE Balanced,RF+XGBoost,0.757857,0.756315,0.563143,0.645589,0.758648,0.727266
9,XGBoost,Original (102 features),RF+XGBoost,0.779270,0.555931,0.804066,0.657362,0.714740,0.743550


In [27]:
# Sort by recall_P (Poor class recall) - primary metric
all_models_sorted = all_models.sort_values('recall_P', ascending=False, na_position='last').reset_index(drop=True)

# Add rank
all_models_sorted.insert(0, 'rank', range(1, len(all_models_sorted) + 1))

# Check if any meet 95% target
meets_target = all_models_sorted['recall_P'] >= 0.95
all_models_sorted['meets_95_target'] = meets_target

print("="*80)
print("ALL MODELS RANKED BY POOR CLASS RECALL")
print("="*80)
print(f"\nModels meeting 95% target: {meets_target.sum()}")
print(f"Best Poor recall: {all_models_sorted['recall_P'].max():.3f}")
if meets_target.sum() == 0:
    print(f"Gap to target: {0.95 - all_models_sorted['recall_P'].max():.3f}")

display(all_models_sorted[['rank', 'model', 'dataset', 'category', 'accuracy', 
                            'recall_P', 'precision_P', 'f1_P', 'meets_95_target']])

# Save
all_models_sorted.to_csv('all_models_combined.csv', index=False)
print("\n✓ Saved: all_models_combined.csv")

ALL MODELS RANKED BY POOR CLASS RECALL

Models meeting 95% target: 0
Best Poor recall: 0.870
Gap to target: 0.080


,rank,model,dataset,category,accuracy,recall_P,precision_P,f1_P,meets_95_target
0,1,Gradient Boosting,Original,Advanced,0.749310,0.870000,0.470000,0.610000,False
1,2,Random Forest,Original (102 features),RF+XGBoost,0.756099,0.869905,0.481388,0.619794,False
2,3,Random Forest,Engineered (130 features),RF+XGBoost,0.754034,0.869569,0.483900,0.621787,False
3,4,Decision Tree,Original,Baseline,0.914759,0.860000,0.850000,0.850000,False
4,5,Random Forest,SMOTE Balanced,RF+XGBoost,0.744366,0.812451,0.486490,0.608571,False
5,6,Random Forest,PCA (74 components),RF+XGBoost,0.736335,0.806069,0.463897,0.588887,False
6,7,XGBoost,SMOTE Balanced,RF+XGBoost,0.757857,0.756315,0.563143,0.645589,False
7,8,Logistic Regression,Original,Baseline,0.647103,0.730000,0.330000,0.460000,False
8,9,Naive Bayes,Original,Baseline,0.482186,0.680000,0.240000,0.350000,False
9,10,Gradient Boosting,SMOTE,Advanced,0.783599,0.660000,0.710000,0.680000,False



✓ Saved: all_models_combined.csv


In [29]:
print("="*80)
print("TOP 10 MODELS BY POOR CLASS RECALL")
print("="*80)

top_10 = all_models_sorted.head(10)

for i, row in top_10.iterrows():
    status = "✓ MEETS TARGET" if row['meets_95_target'] else "✗ Below target"
    print(f"\n#{row['rank']}: {row['model']} - {row['dataset']} ({row['category']})")
    print(f"  Recall (Poor): {row['recall_P']:.3f} {status}")
    print(f"  Precision (Poor): {row['precision_P']:.3f}")
    print(f"  F1 (Poor): {row['f1_P']:.3f}")
    print(f"  Accuracy: {row['accuracy']:.3f}")

TOP 10 MODELS BY POOR CLASS RECALL

#1: Gradient Boosting - Original (Advanced)
  Recall (Poor): 0.870 ✗ Below target
  Precision (Poor): 0.470
  F1 (Poor): 0.610
  Accuracy: 0.749

#2: Random Forest - Original (102 features) (RF+XGBoost)
  Recall (Poor): 0.870 ✗ Below target
  Precision (Poor): 0.481
  F1 (Poor): 0.620
  Accuracy: 0.756

#3: Random Forest - Engineered (130 features) (RF+XGBoost)
  Recall (Poor): 0.870 ✗ Below target
  Precision (Poor): 0.484
  F1 (Poor): 0.622
  Accuracy: 0.754

#4: Decision Tree - Original (Baseline)
  Recall (Poor): 0.860 ✗ Below target
  Precision (Poor): 0.850
  F1 (Poor): 0.850
  Accuracy: 0.915

#5: Random Forest - SMOTE Balanced (RF+XGBoost)
  Recall (Poor): 0.812 ✗ Below target
  Precision (Poor): 0.486
  F1 (Poor): 0.609
  Accuracy: 0.744

#6: Random Forest - PCA (74 components) (RF+XGBoost)
  Recall (Poor): 0.806 ✗ Below target
  Precision (Poor): 0.464
  F1 (Poor): 0.589
  Accuracy: 0.736

#7: XGBoost - SMOTE Balanced (RF+XGBoost)
  Recall 

In [31]:
print("="*80)
print("PERFORMANCE BY CATEGORY")
print("="*80)

for category in ['Baseline', 'RF+XGBoost', 'Advanced']:
    cat_data = all_models_sorted[all_models_sorted['category'] == category]
    
    print(f"\n{category} ({len(cat_data)} models):")
    print(f"  Avg Recall (Poor): {cat_data['recall_P'].mean():.3f}")
    print(f"  Max Recall (Poor): {cat_data['recall_P'].max():.3f}")
    print(f"  Avg Accuracy: {cat_data['accuracy'].mean():.3f}")
    print(f"  Avg F1 (Poor): {cat_data['f1_P'].mean():.3f}")
    
    # Best model in category
    best = cat_data.iloc[0]
    print(f"  Best: {best['model']} - {best['dataset']} (Recall: {best['recall_P']:.3f})")

PERFORMANCE BY CATEGORY

Baseline (4 models):
  Avg Recall (Poor): 0.705
  Max Recall (Poor): 0.860
  Avg Accuracy: 0.716
  Avg F1 (Poor): 0.580
  Best: Decision Tree - Original (Recall: 0.860)

RF+XGBoost (8 models):
  Avg Recall (Poor): 0.699
  Max Recall (Poor): 0.870
  Avg Accuracy: 0.755
  Avg F1 (Poor): 0.611
  Best: Random Forest - Original (102 features) (Recall: 0.870)

Advanced (5 models):
  Avg Recall (Poor): 0.765
  Max Recall (Poor): 0.870
  Avg Accuracy: 0.768
  Avg F1 (Poor): 0.645
  Best: Gradient Boosting - Original (Recall: 0.870)


In [33]:
# Create clean display table
summary = all_models_sorted.copy()

# Format numeric columns
for col in ['accuracy', 'recall_P', 'precision_P', 'f1_P', 'recall_macro', 'f1_macro']:
    if col in summary.columns:
        summary[col] = summary[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")

# Select columns for final table
final_cols = ['rank', 'model', 'dataset', 'category', 'accuracy', 
              'recall_P', 'precision_P', 'f1_P']

final_table = summary[final_cols]

print("="*80)
print("FINAL COMPARATIVE TABLE")
print("="*80)
display(final_table)

# Save
final_table.to_csv('models_comparison_table.csv', index=False)
print("\n✓ Saved: models_comparison_table.csv")

FINAL COMPARATIVE TABLE


,rank,model,dataset,category,accuracy,recall_P,precision_P,f1_P
0,1,Gradient Boosting,Original,Advanced,0.749,0.870,0.470,0.610
1,2,Random Forest,Original (102 features),RF+XGBoost,0.756,0.870,0.481,0.620
2,3,Random Forest,Engineered (130 features),RF+XGBoost,0.754,0.870,0.484,0.622
3,4,Decision Tree,Original,Baseline,0.915,0.860,0.850,0.850
4,5,Random Forest,SMOTE Balanced,RF+XGBoost,0.744,0.812,0.486,0.609
5,6,Random Forest,PCA (74 components),RF+XGBoost,0.736,0.806,0.464,0.589
6,7,XGBoost,SMOTE Balanced,RF+XGBoost,0.758,0.756,0.563,0.646
7,8,Logistic Regression,Original,Baseline,0.647,0.730,0.330,0.460
8,9,Naive Bayes,Original,Baseline,0.482,0.680,0.240,0.350
9,10,Gradient Boosting,SMOTE,Advanced,0.784,0.660,0.710,0.680



✓ Saved: models_comparison_table.csv


In [35]:
# In your analysis, split the comparison:

# Models with complete Poor class metrics
complete_models = all_models_sorted[all_models_sorted['recall_P'].notna() & 
                                     all_models_sorted['precision_P'].notna()]

print("Models with complete Poor class metrics:")
display(complete_models[['rank', 'model', 'dataset', 'recall_P', 'precision_P', 'f1_P']])

# Models with only macro metrics
incomplete_models = all_models_sorted[all_models_sorted['precision_P'].isna()]

print("\nModels with macro metrics only (cannot compare on Poor class):")
display(incomplete_models[['rank', 'model', 'dataset', 'accuracy', 'recall_macro', 'f1_macro']])

Models with complete Poor class metrics:


,rank,model,dataset,recall_P,precision_P,f1_P
0,1,Gradient Boosting,Original,0.870000,0.470000,0.610000
1,2,Random Forest,Original (102 features),0.869905,0.481388,0.619794
2,3,Random Forest,Engineered (130 features),0.869569,0.483900,0.621787
3,4,Decision Tree,Original,0.860000,0.850000,0.850000
4,5,Random Forest,SMOTE Balanced,0.812451,0.486490,0.608571
5,6,Random Forest,PCA (74 components),0.806069,0.463897,0.588887
6,7,XGBoost,SMOTE Balanced,0.756315,0.563143,0.645589
7,8,Logistic Regression,Original,0.730000,0.330000,0.460000
8,9,Naive Bayes,Original,0.680000,0.240000,0.350000
9,10,Gradient Boosting,SMOTE,0.660000,0.710000,0.680000



Models with macro metrics only (cannot compare on Poor class):


,rank,model,dataset,accuracy,recall_macro,f1_macro
14,15,Neural Network,Original,0.776417,0.722620,0.743519
15,16,Neural Network,PCA,0.773516,0.712905,0.737617
16,17,Neural Network,SMOTE,0.758996,0.741793,0.721422


In [51]:
import altair as alt
import pandas as pd

# Load your data
data = pd.read_csv('all_models_combined.csv')

# Filter out models with missing Poor metrics
data_complete = data[data['recall_P'].notna() & data['precision_P'].notna()].copy()

# Create scatter plot
scatter = alt.Chart(data_complete).mark_circle(size=200, opacity=0.8).encode(
    x=alt.X('recall_P:Q', 
            scale=alt.Scale(domain=[0.3, 1.0]),
            title='Recall (Poor Class)',
            axis=alt.Axis(format='.0%')),
    y=alt.Y('precision_P:Q', 
            scale=alt.Scale(domain=[0.2, 1.0]),
            title='Precision (Poor Class)',
            axis=alt.Axis(format='.0%')),
    color=alt.Color('category:N',
                    scale=alt.Scale(domain=['Baseline', 'RF+XGBoost', 'Advanced'],
                                   range=['#4CAF50', '#2196F3', '#FF9800']),
                    legend=alt.Legend(title='Model Category')),
    tooltip=['model:N', 'dataset:N', 'recall_P:Q', 'precision_P:Q', 'f1_P:Q', 'accuracy:Q']
).properties(
    width=600,
    height=400,
    title='Recall-Precision Trade-off: Poor Class Performance'
)

# Add 95% recall target line
target_line = alt.Chart(pd.DataFrame({'x': [0.95]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(
    x='x:Q'
)

# Add reference lines for quadrants
ideal_zone = alt.Chart(pd.DataFrame({
    'recall': [0.95, 0.95, 1.0, 1.0],
    'precision': [0.7, 1.0, 1.0, 0.7]
})).mark_rect(opacity=0.1, color='green').encode(
    x='recall:Q',
    y='precision:Q',
    x2=alt.X2(datum=1.0),
    y2=alt.Y2(datum=1.0)
)

# Combine
viz1 = ideal_zone + scatter + target_line

viz1.save('recall_precision_tradeoff.html')
print("✓ Saved: recall_precision_tradeoff.html")
viz1

✓ Saved: recall_precision_tradeoff.html


C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.

alt.LayerChart(...)

In [53]:
# Prepare data for heatmap
heatmap_data = data_complete.copy()

# Select top 12 models
heatmap_data = heatmap_data.head(12)

# Create metric columns for visualization
metrics_long = heatmap_data.melt(
    id_vars=['rank', 'model', 'dataset'],
    value_vars=['recall_P', 'precision_P', 'f1_P', 'accuracy'],
    var_name='metric',
    value_name='score'
)

# Rename metrics for display
metric_labels = {
    'recall_P': 'Recall (Poor)',
    'precision_P': 'Precision (Poor)',
    'f1_P': 'F1 (Poor)',
    'accuracy': 'Accuracy'
}
metrics_long['metric'] = metrics_long['metric'].map(metric_labels)

# Create model labels
metrics_long['model_label'] = metrics_long.apply(
    lambda x: f"#{int(x['rank'])} {x['model'][:15]} - {x['dataset'][:8]}", axis=1
)

# Create heatmap
heatmap = alt.Chart(metrics_long).mark_rect().encode(
    x=alt.X('metric:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('model_label:N', title='Model', sort=None),
    color=alt.Color('score:Q',
                    scale=alt.Scale(scheme='viridis', domain=[0, 1]),
                    legend=alt.Legend(title='Score', format='.0%')),
    tooltip=['model:N', 'dataset:N', 'metric:N', 
             alt.Tooltip('score:Q', format='.3f')]
).properties(
    width=400,
    height=500,
    title='Top 12 Models: Performance Heatmap'
)

# Add text labels
text = alt.Chart(metrics_long).mark_text(baseline='middle').encode(
    x='metric:N',
    y=alt.Y('model_label:N', sort=None),
    text=alt.Text('score:Q', format='.2f'),
    color=alt.condition(
        alt.datum.score > 0.6,
        alt.value('white'),
        alt.value('black')
    )
)

viz2 = heatmap + text

viz2.save('performance_heatmap.html')
print("✓ Saved: performance_heatmap.html")
viz2

✓ Saved: performance_heatmap.html


C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.

alt.LayerChart(...)

In [55]:
# Calculate category averages
category_stats = data_complete.groupby('category').agg({
    'recall_P': 'mean',
    'precision_P': 'mean'
}).reset_index()

category_stats = category_stats.sort_values('recall_P', ascending=False)

# Create grouped bar chart
bars = alt.Chart(category_stats).mark_bar(size=80).encode(
    x=alt.X('category:N', title='Model Category', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('recall_P:Q', 
            scale=alt.Scale(domain=[0, 1]),
            title='Average Recall (Poor Class)',
            axis=alt.Axis(format='.0%')),
    color=alt.Color('category:N',
                    scale=alt.Scale(domain=['Baseline', 'RF+XGBoost', 'Advanced'],
                                   range=['#4CAF50', '#2196F3', '#FF9800']),
                    legend=None),
    tooltip=['category:N', 
             alt.Tooltip('recall_P:Q', title='Avg Recall', format='.3f'),
             alt.Tooltip('precision_P:Q', title='Avg Precision', format='.3f')]
).properties(
    width=400,
    height=300,
    title='Average Recall by Model Category'
)

# Add target line
target = alt.Chart(pd.DataFrame({'y': [0.95]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(y='y:Q')

# Add value labels
text = alt.Chart(category_stats).mark_text(
    align='center',
    baseline='bottom',
    dy=-5,
    fontSize=14,
    fontWeight='bold'
).encode(
    x='category:N',
    y='recall_P:Q',
    text=alt.Text('recall_P:Q', format='.1%')
)

viz3 = bars + target + text

viz3.save('recall_by_category.html')
print("✓ Saved: recall_by_category.html")
viz3

✓ Saved: recall_by_category.html


C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.

alt.LayerChart(...)

In [57]:
# Top 15 models
top_15 = data_complete.head(15).copy()

# Create ranking chart
base = alt.Chart(top_15).encode(
    y=alt.Y('rank:O', title='Rank', sort='ascending')
)

# Recall bars
recall_bars = base.mark_bar(color='#2196F3', opacity=0.7).encode(
    x=alt.X('recall_P:Q', 
            scale=alt.Scale(domain=[0, 1]),
            title='Performance Score',
            axis=alt.Axis(format='.0%')),
    tooltip=['rank:O', 'model:N', 'dataset:N', 
             alt.Tooltip('recall_P:Q', title='Recall', format='.3f')]
)

# F1 points
f1_points = base.mark_circle(size=100, color='#FF9800').encode(
    x='f1_P:Q',
    tooltip=['rank:O', 'model:N', 
             alt.Tooltip('f1_P:Q', title='F1 Score', format='.3f')]
)

# Model labels
labels = base.mark_text(align='left', dx=5, fontSize=10).encode(
    x=alt.value(10),
    text=alt.Text('model:N')
)

# Target line
target = alt.Chart(pd.DataFrame({'x': [0.95]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(x='x:Q')

viz4 = (recall_bars + f1_points + target).properties(
    width=600,
    height=500,
    title='Model Rankings: Recall (bars) vs F1 (points)'
)

viz4.save('model_rankings.html')
print("✓ Saved: model_rankings.html")
viz4

✓ Saved: model_rankings.html


C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.

alt.LayerChart(...)

In [59]:
# Filter RF+XGBoost models
rf_xgb_only = data_complete[data_complete['category'] == 'RF+XGBoost'].copy()

# Extract model type
rf_xgb_only['model_type'] = rf_xgb_only['model'].apply(
    lambda x: 'Random Forest' if 'Random Forest' in x else 'XGBoost'
)

# Create grouped chart
chart = alt.Chart(rf_xgb_only).mark_bar().encode(
    x=alt.X('dataset:N', title='Dataset', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('recall_P:Q', 
            scale=alt.Scale(domain=[0, 1]),
            title='Recall (Poor Class)',
            axis=alt.Axis(format='.0%')),
    color=alt.Color('model_type:N', 
                    scale=alt.Scale(domain=['Random Forest', 'XGBoost'],
                                   range=['#2196F3', '#FF5722']),
                    legend=alt.Legend(title='Model Type')),
    xOffset='model_type:N',
    tooltip=['model_type:N', 'dataset:N', 
             alt.Tooltip('recall_P:Q', format='.3f'),
             alt.Tooltip('precision_P:Q', format='.3f'),
             alt.Tooltip('accuracy:Q', format='.3f')]
).properties(
    width=500,
    height=350,
    title='Dataset Impact: RF vs XGBoost Performance'
)

# Add target line
target = alt.Chart(pd.DataFrame({'y': [0.95]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(y='y:Q')

viz5 = chart + target

viz5.save('dataset_comparison.html')
print("✓ Saved: dataset_comparison.html")
viz5

✓ Saved: dataset_comparison.html


C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\vashi\anaconda3\Lib\site-packages\altair\utils\core.

alt.LayerChart(...)